# Unified Workflow Recommendation API Server

This notebook loads the trained Deberta classification model and Flan-T5 step recommendation model, starts a FastAPI application, and opens a public ngrok tunnel so that `workflow_service` can call the AI recommendations.

In [ ]:
# 1. Install required dependencies
!pip install -q fastapi uvicorn pyngrok nest_asyncio transformers torch sentencepiece protobuf

In [ ]:
# 2. Initialize and define API application
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional
from pyngrok import ngrok
import torch
from transformers import AutoModelForSequenceClassification, AutoModelForSeq2SeqLM, AutoTokenizer
import os

nest_asyncio.apply()
app = FastAPI(title="Unified Kaggle Workflow Recommendation Service")

# ── Classification model: load d90nqm/contract-workflow directly from HuggingFace ──
DEBERTA_HF_ID = "d90nqm/contract-workflow"
print(f"Loading classification model from HuggingFace: {DEBERTA_HF_ID}...")
try:
    deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_HF_ID)
    deberta_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_HF_ID)
    deberta_model.eval()
    if torch.cuda.is_available():
        deberta_model.to("cuda")
    print(f"Classification model loaded! Labels: {list(deberta_model.config.id2label.values())}")
except Exception as e:
    print(f"Failed to load classification model: {e}")
    deberta_model = None

# ── Step generation model: Flan-T5 fine-tuned locally on Kaggle ──────────────
FLANT5_PATH = "Doan2108/dynamic_worflow"
if not os.path.exists(FLANT5_PATH):
    FLANT5_PATH = "google/flan-t5-base"  # fallback to base model if not yet trained

print(f"Loading Flan-T5 step builder from {FLANT5_PATH}...")
try:
    flant5_tokenizer = AutoTokenizer.from_pretrained(FLANT5_PATH)
    flant5_model = AutoModelForSeq2SeqLM.from_pretrained(FLANT5_PATH)
    flant5_model.eval()
    if torch.cuda.is_available():
        flant5_model.to("cuda")
    print("Flan-T5 model loaded successfully!")
except Exception as e:
    print(f"Failed to load Flan-T5: {e}")
    flant5_model = None

# Define detailed step library mapping (descriptions and role ids)
STEP_DETAILS_MAPPING = {
    "Contract Negotiation": {"role_id": 4, "description": "Thương thảo các điều khoản chưa thống nhất giữa các bên ký kết."},
    "Legal Review": {"role_id": 4, "description": "Rà soát tính pháp lý, rủi ro điều khoản và tuân thủ pháp luật."},
    "Technical Review": {"role_id": 7, "description": "Thẩm định tính khả thi kỹ thuật và giải pháp công nghệ đề xuất."},
    "Security Review": {"role_id": 8, "description": "Đánh giá an toàn thông tin, bảo mật dữ liệu và hệ thống."},
    "Compliance Review": {"role_id": 9, "description": "Kiểm tra sự tuân thủ các quy định nội bộ và tiêu chuẩn ngành."},
    "Finance Review": {"role_id": 6, "description": "Thẩm định ngân sách, dòng tiền và nghĩa vụ tài chính phát sinh."},
    "Procurement Review": {"role_id": 10, "description": "Đánh giá năng lực nhà cung cấp, đơn giá và chính sách mua sắm."},
    "Manager Approval": {"role_id": 5, "description": "Phê duyệt cấp quản lý trực tiếp về mặt chủ trương và ngân sách."},
    "Director Approval": {"role_id": 11, "description": "Phê duyệt cấp Giám đốc bộ phận đối với các hợp đồng/dự án lớn."},
    "Executive Approval": {"role_id": 11, "description": "Phê duyệt tối cao từ Ban điều hành/Tổng giám đốc."},
    "Contract Signing": {"role_id": 4, "description": "Đại diện có thẩm quyền thực hiện ký kết hợp đồng chính thức."},
    "Document Archive": {"role_id": 4, "description": "Lưu trữ hợp đồng đã ký kết vào hệ thống và bàn giao bản cứng."}
}

INDEX_TO_WORKFLOW_ID = {
    0: 'WF_EMPLOYMENT',
    1: 'WF_EXECUTIVE',
    2: 'WF_GENERAL',
    3: 'WF_NDA',
    4: 'WF_PROCUREMENT',
    5: 'WF_PURCHASE',
    6: 'WF_SERVICE',
    7: 'WF_VENDOR'
}

class RecommendWorkflowRequest(BaseModel):
    contract_text: str
    clause_types: List[str] = []
    contract_type: str = ""

class WorkflowStepResponse(BaseModel):
    step_name: str
    role_id: int
    description: str

class RecommendWorkflowResponse(BaseModel):
    workflow_type: str
    steps: List[WorkflowStepResponse]
    reasons: str
    workflow_name: Optional[str] = None

@app.post("/api/v1/recommend_workflow", response_model=RecommendWorkflowResponse)
async def recommend_workflow_api(payload: RecommendWorkflowRequest):
    # 1. Classification
    workflow_type = "WF_GENERAL"
    workflow_reason = "Quy trình phê duyệt hợp đồng tiêu chuẩn."
    
    if deberta_model is not None:
        try:
            device = "cuda" if torch.cuda.is_available() else "cpu"
            inputs = deberta_tokenizer(
                payload.contract_text,
                return_tensors="pt",
                truncation=True,
                max_length=512
            ).to(device)
            with torch.no_grad():
                outputs = deberta_model(**inputs)
            pred_idx = torch.argmax(outputs.logits, dim=-1).item()
            if hasattr(deberta_model.config, 'id2label') and deberta_model.config.id2label:
                workflow_type = deberta_model.config.id2label.get(pred_idx, "WF_GENERAL")
            else:
                workflow_type = INDEX_TO_WORKFLOW_ID.get(pred_idx, "WF_GENERAL")

            if flant5_model is not None:
                wf_prompt = (
                    f"Based on the contract context: '{payload.contract_text[:300]}...', "
                    f"explain why this contract is classified as '{workflow_type}' in 1 short Vietnamese sentence."
                )
                wf_inputs = flant5_tokenizer(wf_prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
                
                with torch.no_grad():
                    wf_outputs = flant5_model.generate(**wf_inputs, max_new_tokens=128)
                    
                generated_wf_reason = flant5_tokenizer.decode(wf_outputs[0], skip_special_tokens=True).strip()
                
                if len(generated_wf_reason) >= 5:
                    workflow_reason = generated_wf_reason
        except Exception as e:
            print(f"Classification error: {e}")

    # 2. Step Generation
    steps_list = []
    reasons = workflow_reason
    if flant5_model is not None:
        try:
            device = "cuda" if torch.cuda.is_available() else "cpu"
            prompt = "Generate the contract workflow steps for: " + payload.contract_text
            inputs = flant5_tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=512
            ).to(device)
            with torch.no_grad():
                outputs = flant5_model.generate(**inputs, max_new_tokens=64)
            decoded = flant5_tokenizer.decode(outputs[0], skip_special_tokens=True)
            parsed_steps = [s.strip() for s in decoded.split("->") if s.strip()]
            
            for step_name in parsed_steps:
                if step_name in STEP_DETAILS_MAPPING:
                    details = STEP_DETAILS_MAPPING[step_name]
                    context = payload.contract_text[:300].replace("\n", " ").strip()

                    desc_prompt = (
                        f"Based on the contract context: '{context}', "
                        f"explain the contract approval step '{step_name}' in 1 short sentence in Vietnamese."
                    )
                    desc_inputs = flant5_tokenizer(desc_prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
                    with torch.no_grad():
                        desc_outputs = flant5_model.generate(**desc_inputs, max_new_tokens=64)
                    dynamic_desc = flant5_tokenizer.decode(desc_outputs[0], skip_special_tokens=True).strip()
                    
                    if len(dynamic_desc) < 5:
                        dynamic_desc = details["description"]

                    steps_list.append(WorkflowStepResponse(
                        step_name=step_name,
                        role_id=details["role_id"],
                        description=dynamic_desc
                    ))
        except Exception as e:
            print(f"Step generation error: {e}")

    # Fallback to defaults if empty
    if not steps_list:
        default_steps = ["Legal Review", "Contract Signing", "Document Archive"]
        for step_name in default_steps:
            details = STEP_DETAILS_MAPPING[step_name]
            steps_list.append(WorkflowStepResponse(
                step_name=step_name,
                role_id=details["role_id"],
                description=details["description"]
            ))

    workflow_name = f"{workflow_type.replace('WF_', '').title()} Approval Workflow"
    return RecommendWorkflowResponse(
        workflow_type=workflow_type,
        steps=steps_list,
        reasons=reasons,
        workflow_name=workflow_name
    )

In [ ]:
# 3. Set Ngrok Token and Start Uvicorn Server
# Fix: Kaggle/Jupyter already has a running event loop.
# Use uvicorn.Server + await instead of uvicorn.run() to avoid RuntimeError.
import asyncio
nest_asyncio.apply()  # allow nested event loops in Jupyter

NGROK_TOKEN = "3Falm90byk0kDTynqQaVBJpyMOn_7QGUtEZT13zMKCej5qq65"
ngrok.set_auth_token(NGROK_TOKEN)
try:
    tunnel = ngrok.connect(8000)
    print("\n==============================")
    print("  NGROK PUBLIC URL:")
    print(f"  {tunnel.public_url}")
    print("  Copy this URL to KAGGLE_AI_URL")
    print("==============================\n")
except Exception as e:
    print(f"Ngrok tunnel failed: {e}")

# Start server using await (compatible with Kaggle/Jupyter event loop)
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()